# LangGraph Learning Path

This notebook provides a unified, progressive learning path through the core concepts of **LangGraph** — the framework for building stateful, multi-step AI agent workflows on top of LangChain.

## What You Will Learn

| Section | Topic |
|---------|-------|
| 1 | Graph Components: States, Nodes, and Edges |
| 2 | Building Your First LangGraph Graph |
| 3 | Conditional Edges and Routing Functions |
| 4 | The Annotated Construct and Reducer Functions |
| 5 | Reducer Functions in Action |
| 6 | The MessagesState Class |
| 7 | The RemoveMessage Class |
| 8 | Trimming Messages |
| 9 | Summarizing Messages |
| 10 | Short-Term Memory with InMemorySaver |
| 11 | The StateSnapshot Class |
| 12 | Long-Term Memory with SQLite |

## Prerequisites
- Python 3.11+
- A `.env` file with `OPENAI_API_KEY=your_key_here` in the same directory as this notebook
- All dependencies installed (see README.md)

---
## Setup: Environment and Deprecation Suppression

We suppress deprecation warnings first so the output stays clean throughout the notebook.
Then we load environment variables from the `.env` file using `python-dotenv`.

> **Note:** The `%load_ext dotenv` / `%dotenv` magic approach used in the original course files
> requires the `ipython-dotenv` extension which may not be installed in all environments.
> We use `python-dotenv` directly here for broader compatibility.

In [ ]:
# Suppress deprecation warnings to keep output clean.
# LangChain evolves rapidly and some older patterns trigger warnings
# that are informational but distracting during learning.
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Load environment variables from .env file.
# Your .env file should contain: OPENAI_API_KEY=sk-...
# python-dotenv reads this file and sets the variables in os.environ
# so that LangChain / OpenAI libraries can pick them up automatically.
from dotenv import load_dotenv
load_dotenv()

print("Environment loaded successfully.")

---
## Section 1 — Graph Components: Importing Relevant Classes

### Core Concepts

LangGraph is built around three fundamental primitives that map to graph theory:

- **State** — A typed dictionary that represents the current snapshot of data flowing through the graph. Every node reads from and writes to this shared state object.
- **Nodes** — Python functions (or runnables) that perform a unit of work. Each node receives the current state and returns an updated state.
- **Edges** — The connections that determine which node runs next. Edges can be static (always go to node X) or conditional (choose a node based on the current state).

### Special Nodes: START and END
- `START` is a virtual entry point — it signals where the graph execution begins.
- `END` is a virtual exit point — it signals where a particular path terminates.

### StateGraph
`StateGraph` is the main class used to define and compile a graph. You pass it your State schema, then add nodes and edges before compiling it into an executable runnable.

In [ ]:
# Core LangGraph imports:
#   START  - virtual start node (entry point marker)
#   END    - virtual end node (exit point marker)
#   StateGraph - the graph builder class
from langgraph.graph import START, END, StateGraph

# TypedDict gives us a typed dictionary class for defining State schemas.
# Using TypedDict instead of a plain dict enables static type checking and
# makes the structure of the state explicit and self-documenting.
from typing_extensions import TypedDict

# ChatOpenAI is the LangChain wrapper around OpenAI's chat completion API.
# It handles authentication, request formatting, and response parsing.
from langchain_openai.chat_models import ChatOpenAI

# Message types:
#   HumanMessage  - represents a message from the user
#   BaseMessage   - the base class for all message types (used in type hints)
from langchain_core.messages import HumanMessage, BaseMessage

# Runnable is the base interface for LangChain/LangGraph components.
# A compiled graph IS a Runnable, so you can call .invoke(), .stream(), etc.
from langchain_core.runnables import Runnable

# Sequence is a generic type hint for ordered collections (lists, tuples).
# We use it in State definitions to say "messages is a list of BaseMessage objects".
from collections.abc import Sequence

print("All imports successful.")

---
## Section 2 — Building Your First Graph

### Defining the State

The State is the backbone of any LangGraph application. It defines **what information persists** and gets passed between nodes.

Here we define a minimal state with a single field: `messages`, which holds a sequence of chat messages. As the graph executes, nodes read from and write to this `messages` list.

> **Important:** In this early example, each node *replaces* the messages list entirely (returning `State(messages=[...])` overwrites it). In later sections we will use the `add_messages` reducer to *append* instead of overwrite.

In [ ]:
# Define the State schema.
# TypedDict creates a dictionary class where each key has a declared type.
# LangGraph will validate that nodes return dictionaries matching this schema.
#
# Here, 'messages' is typed as Sequence[BaseMessage], meaning:
#   - it must be ordered (list, tuple, etc.)
#   - each element must be a BaseMessage or a subclass (HumanMessage, AIMessage, etc.)
class State(TypedDict):
    messages: Sequence[BaseMessage]

In [ ]:
# Create a test state instance with a single human message.
# This demonstrates how the state looks at the start of a conversation.
# HumanMessage wraps a string into the correct message format for the LLM.
state = State(messages=[HumanMessage("Could you tell me a grook by Piet Hein?")])

# pretty_print() is a convenience method on all message types that formats
# the message with a clear Human/AI/System prefix for easy reading.
state["messages"][0].pretty_print()

### Defining the Chatbot Node

A **node** in LangGraph is simply a Python function that:
1. Accepts the current `State` as its only argument
2. Performs some work (in this case, calling an LLM)
3. Returns a partial or full `State` update

The `ChatOpenAI` model is configured with:
- `model="gpt-4o"` — use the GPT-4o model
- `seed=365` — fixed random seed for reproducible outputs
- `temperature=0` — deterministic (no randomness in token selection)
- `max_completion_tokens=100` — cap the response length

In [ ]:
# Instantiate the LLM. This object is reused across multiple nodes.
# Setting seed and temperature=0 makes responses deterministic and reproducible,
# which is ideal when learning — you'll see the same output each run.
#
# max_completion_tokens=100 keeps costs low during development.
# NOTE: 'max_completion_tokens' is the current parameter name (replaces
# the deprecated 'max_tokens' in newer versions of the openai library).
chat = ChatOpenAI(
    model="gpt-4o",
    seed=365,
    temperature=0,
    max_completion_tokens=100
)

In [ ]:
# Test the model directly (outside the graph) to confirm connectivity.
# chat.invoke() sends the list of messages to the API and returns an AIMessage.
response = chat.invoke(state["messages"])
response.pretty_print()

In [ ]:
# Define the chatbot node function.
# Node functions must accept a State and return a State (or a dict matching the State schema).
#
# The print statement traces execution flow — useful for understanding
# which nodes are being visited as the graph runs.
def chatbot(state: State) -> State:
    print("\n-------> ENTERING chatbot:")
    
    # Pass the full message history to the LLM.
    # The model uses the entire list as context.
    response = chat.invoke(state["messages"])
    response.pretty_print()
    
    # Return a new State where messages is replaced with just the AI response.
    # NOTE: In this simple version without reducers, returning [response] 
    # overwrites the previous messages list. We improve this in Section 4.
    return State(messages=[response])

In [ ]:
# Test the node function directly (before wiring it into a graph).
# This confirms the function works in isolation.
chatbot(state)

### Building and Compiling the Graph

Once we have a State and nodes, we wire everything together using `StateGraph`:

1. `add_node(name, function)` — register a node with a label
2. `add_edge(from, to)` — connect two nodes with a directed edge
3. `compile()` — validate the graph and return an executable `Runnable`

The compiled graph can be invoked just like any other LangChain runnable.

In [ ]:
# Create a new StateGraph, passing our State schema.
# LangGraph uses the schema to validate node inputs/outputs.
graph = StateGraph(State)

# Register nodes. The string name is used when adding edges and in debug output.
graph.add_node("chatbot", chatbot)

# Add edges to define the execution path.
#   START -> chatbot: the graph begins by entering the chatbot node
#   chatbot -> END:   after chatbot runs, the graph terminates
graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)

In [ ]:
# Compile the graph into a runnable object.
# compile() validates the graph structure (e.g., checks that all edges connect
# to registered nodes) and returns an object you can call .invoke() on.
graph_compiled = graph.compile()

# Verify that the compiled graph implements the Runnable interface.
# This confirms it can be used with .invoke(), .stream(), .batch(), etc.
print(f"Is Runnable: {isinstance(graph_compiled, Runnable)}")

In [ ]:
# Display the compiled graph visually in the notebook.
# Jupyter renders the graph object as a diagram showing nodes and edges.
graph_compiled

In [ ]:
# Run the graph end-to-end.
# .invoke() sends the initial state through the graph and returns the final state.
# The output is the State dictionary as it exists when END is reached.
result = graph_compiled.invoke(state)
result

---
## Section 3 — Conditional Edges and Routing Functions

### Why Conditional Edges?

Static edges always go to the same next node. But real applications need **branching logic** — for example: *"if the user wants to continue, loop back; otherwise, end."*

Conditional edges solve this. Instead of a fixed destination, you provide a **routing function** that inspects the current state and returns a string indicating which node to visit next.

### New Nodes in This Section
- `ask_question` — prompts the user to type a question (using `input()`)
- `chatbot` — sends the question to the LLM and prints the response
- `ask_another_question` — asks whether the user wants to continue

### The Routing Function
The routing function reads the last message in state and returns either `"ask_question"` (loop back) or `"__end__"` (stop the graph).

> **Note:** The string `"__end__"` is LangGraph's internal name for the END sentinel. You can also return the `END` constant directly.

In [ ]:
# Additional import needed for conditional edges:
#   Literal - restricts the return type of the routing function to a fixed set of strings.
#             This is important for type safety and LangGraph's internal validation.
from typing import Literal

In [ ]:
# Node: ask_question
# This node uses Python's built-in input() to collect a question from the user.
# In a production system, this would be replaced by a message from a UI or API.
#
# It wraps the user's typed text in a HumanMessage and stores it in the state.
def ask_question(state: State) -> State:
    print("\n-------> ENTERING ask_question:")
    print("What is your question?")
    return State(messages=[HumanMessage(input())])


# Node: chatbot
# Sends the current messages to the LLM and returns the AI response.
def chatbot(state: State) -> State:
    print("\n-------> ENTERING chatbot:")
    response = chat.invoke(state["messages"])
    response.pretty_print()
    return State(messages=[response])


# Node: ask_another_question
# After the chatbot responds, this node asks if the user wants to continue.
# The user's yes/no answer is stored as a HumanMessage in the state.
def ask_another_question(state: State) -> State:
    print("\n-------> ENTERING ask_another_question:")
    print("Would you like to ask one more question (yes/no)?")
    return State(messages=[HumanMessage(input())])

In [ ]:
# The Routing Function
# -------------------
# This function is NOT a node — it's a decision function used with add_conditional_edges().
#
# It receives the current state and returns a string that LangGraph uses to
# look up which node to visit next.
#
# Return type annotation Literal["ask_question", "__end__"] is important:
# - It documents the possible paths for readers
# - LangGraph uses it internally to validate the graph structure at compile time
#
# BUG FIX from original: state["messages"].content was incorrect — messages is a list,
# not a single message. The correct access pattern is state["messages"][0].content
# to read the first (and only) message that was just stored by ask_another_question.
def routing_function(state: State) -> Literal["ask_question", "__end__"]:
    if state["messages"][0].content == "yes":
        return "ask_question"
    else:
        return "__end__"

In [ ]:
# Build the conditional graph.
graph = StateGraph(State)

# Register all three nodes.
graph.add_node("ask_question", ask_question)
graph.add_node("chatbot", chatbot)
graph.add_node("ask_another_question", ask_another_question)

# Static edges define the primary flow:
#   START -> ask_question -> chatbot -> ask_another_question
graph.add_edge(START, "ask_question")
graph.add_edge("ask_question", "chatbot")
graph.add_edge("chatbot", "ask_another_question")

# Conditional edge: after ask_another_question, call routing_function to decide next step.
# LangGraph calls routing_function(state) and uses the return value as the next node name.
graph.add_conditional_edges(
    source="ask_another_question",
    path=routing_function
)

graph_compiled = graph.compile()

In [ ]:
# Display the compiled graph. Notice the conditional branch now shows two exit paths.
graph_compiled

In [ ]:
# ASCII visualization — useful when the notebook diagram isn't rendering
# or when working in a terminal environment.
print(graph_compiled.get_graph().draw_ascii())

In [ ]:
# Run the interactive graph.
# The graph will prompt you for questions in the Jupyter cell output area.
# Type a question, press Enter. Then type 'yes' to loop or anything else to stop.
graph_compiled.invoke(State(messages=[]))

---
## Section 4 — The Annotated Construct and Reducer Functions

### The Problem with Overwriting State

In the previous sections, every node returned a new `State(messages=[...])` which **replaced** the entire messages list. This means the graph lost history with each step.

To build a true conversational chatbot, we need messages to **accumulate** across nodes.

### The Solution: Reducers with `Annotated`

Python's `Annotated` type lets you attach metadata to a type hint. LangGraph uses this to attach a **reducer function** to a state field.

A **reducer** is called by LangGraph whenever a node returns an update to a field. Instead of replacing the field, LangGraph calls:
```python
new_value = reducer(old_value, returned_value)
```

### The `add_messages` Reducer

`add_messages` is LangGraph's built-in reducer for message lists. It:
- Appends new messages to the existing list
- Deduplicates by message ID (so replaying the same message won't create duplicates)
- Supports `RemoveMessage` objects for selective deletion (covered in Section 7)

In [ ]:
# Import add_messages — LangGraph's built-in reducer for message accumulation.
# Also import AIMessage to represent assistant responses explicitly.
# Annotated (from typing) lets us attach the reducer to the State field type hint.
from langgraph.graph import add_messages
from langchain_core.messages import AIMessage
from typing import Annotated

In [ ]:
# Demonstrate add_messages directly.
# add_messages takes two sequences of messages and merges them.
# The second argument's messages are appended to the first.
#
# This is exactly what happens internally when a node returns
# State(messages=[new_message]) and the state has the add_messages reducer.
my_list = add_messages(
    [HumanMessage("Hi! I'm Oscar."), AIMessage("Hey, Oscar. How can I assist you?")],
    [HumanMessage("Could you summarize today's news?")]
)

# The result is a single list with all three messages in order.
for msg in my_list:
    msg.pretty_print()

In [ ]:
# Updated State with the add_messages reducer.
#
# Annotated[Sequence[BaseMessage], add_messages] means:
#   - The field holds a Sequence of BaseMessage objects
#   - When a node returns an update to 'messages', LangGraph calls
#     add_messages(current_messages, returned_messages) instead of replacing
#
# This single change transforms our stateless chatbot into a stateful one
# that accumulates conversation history automatically.
class State(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

In [ ]:
# Rebuild the graph with the new accumulating State.
# Note: the node functions remain unchanged — only the State definition changed.
# LangGraph handles the reducer logic transparently.

def ask_question(state: State) -> State:
    print("\n-------> ENTERING ask_question:")
    print("What is your question?")
    return State(messages=[HumanMessage(input())])

def chatbot(state: State) -> State:
    print("\n-------> ENTERING chatbot:")
    response = chat.invoke(state["messages"])
    response.pretty_print()
    return State(messages=[response])

def ask_another_question(state: State) -> State:
    print("\n-------> ENTERING ask_another_question:")
    print("Would you like to ask one more question (yes/no)?")
    return State(messages=[HumanMessage(input())])

def routing_function(state: State) -> Literal["ask_question", "__end__"]:
    # With the add_messages reducer, messages accumulate.
    # We check the LAST message ([-1]) to get the user's most recent answer.
    if state["messages"][-1].content == "yes":
        return "ask_question"
    else:
        return "__end__"

graph = StateGraph(State)
graph.add_node("ask_question", ask_question)
graph.add_node("chatbot", chatbot)
graph.add_node("ask_another_question", ask_another_question)
graph.add_edge(START, "ask_question")
graph.add_edge("ask_question", "chatbot")
graph.add_edge("chatbot", "ask_another_question")
graph.add_conditional_edges(source="ask_another_question", path=routing_function)

graph_compiled = graph.compile()
graph_compiled

In [ ]:
# Run the graph. Now messages accumulate — the LLM sees the full conversation history.
graph_compiled.invoke(State(messages=[]))

---
## Section 5 — Reducer Functions in Action

### Observing State Accumulation

In this section we add print statements inside each node to **visually trace** how the state evolves. This makes it clear that with `add_messages`, each node sees the **complete message history** — not just the messages it personally added.

We also update the node functions to add `AIMessage` objects for the system prompts ("What is your question?" etc.) so these questions also appear in the conversation history passed to the LLM.

In [ ]:
# Updated nodes that print the full accumulated state on entry.
# The state.messages list grows with every node — you can observe this in the output.

def ask_question(state: State) -> State:
    print("\n-------> ENTERING ask_question:")
    # Print every message in the current state to show what has accumulated so far.
    for msg in state["messages"]:
        msg.pretty_print()
    
    question = "What is your question?"
    print(question)
    
    # Return BOTH the AI prompt text and the user's response as new messages.
    # Because of add_messages, these are APPENDED to the existing list.
    return State(messages=[AIMessage(question), HumanMessage(input())])


def chatbot(state: State) -> State:
    print("\n-------> ENTERING chatbot:")
    # The chatbot now sees the FULL accumulated message history.
    # This means the LLM has context from all previous exchanges.
    for msg in state["messages"]:
        msg.pretty_print()
    
    response = chat.invoke(state["messages"])
    response.pretty_print()
    return State(messages=[response])


def ask_another_question(state: State) -> State:
    print("\n-------> ENTERING ask_another_question:")
    for msg in state["messages"]:
        msg.pretty_print()
    
    question = "Would you like to ask one more question (yes/no)?"
    print(question)
    return State(messages=[AIMessage(question), HumanMessage(input())])


def routing_function(state: State) -> Literal["ask_question", "__end__"]:
    # state["messages"][-1] is always the most recent message (the user's yes/no).
    if state["messages"][-1].content == "yes":
        return "ask_question"
    else:
        return "__end__"


graph = StateGraph(State)
graph.add_node("ask_question", ask_question)
graph.add_node("chatbot", chatbot)
graph.add_node("ask_another_question", ask_another_question)
graph.add_edge(START, "ask_question")
graph.add_edge("ask_question", "chatbot")
graph.add_edge("chatbot", "ask_another_question")
graph.add_conditional_edges(source="ask_another_question", path=routing_function)

graph_compiled = graph.compile()

# Run and observe how the message list grows with each loop iteration.
graph_compiled.invoke(State(messages=[]))

---
## Section 6 — The MessagesState Class

### A Convenient Shortcut

Defining `messages: Annotated[Sequence[BaseMessage], add_messages]` is so common in LangGraph applications that LangGraph provides a built-in shortcut: **`MessagesState`**.

`MessagesState` is a pre-built `TypedDict` subclass that already includes the `messages` field with the `add_messages` reducer. Using it eliminates boilerplate and is the recommended pattern for most chat applications.

You can also **extend** `MessagesState` to add your own custom fields (as we will do in Sections 9–12 with the `summary` field).

In [ ]:
# Import MessagesState — the pre-built state class for conversational graphs.
# This is equivalent to:
#   class State(TypedDict):
#       messages: Annotated[list[BaseMessage], add_messages]
from langgraph.graph import MessagesState

In [ ]:
# All node functions now use MessagesState instead of our custom State.
# The behavior is identical — MessagesState already has add_messages built in.

def ask_question(state: MessagesState) -> MessagesState:
    print("\n-------> ENTERING ask_question:")
    for msg in state["messages"]:
        msg.pretty_print()
    question = "What is your question?"
    print(question)
    return MessagesState(messages=[AIMessage(question), HumanMessage(input())])


def chatbot(state: MessagesState) -> MessagesState:
    print("\n-------> ENTERING chatbot:")
    for msg in state["messages"]:
        msg.pretty_print()
    response = chat.invoke(state["messages"])
    response.pretty_print()
    return MessagesState(messages=[response])


def ask_another_question(state: MessagesState) -> MessagesState:
    print("\n-------> ENTERING ask_another_question:")
    for msg in state["messages"]:
        msg.pretty_print()
    question = "Would you like to ask one more question (yes/no)?"
    print(question)
    return MessagesState(messages=[AIMessage(question), HumanMessage(input())])


def routing_function(state: MessagesState) -> Literal["ask_question", "__end__"]:
    if state["messages"][-1].content == "yes":
        return "ask_question"
    else:
        return "__end__"


# Build the graph using MessagesState — note the StateGraph also gets MessagesState.
graph = StateGraph(MessagesState)
graph.add_node("ask_question", ask_question)
graph.add_node("chatbot", chatbot)
graph.add_node("ask_another_question", ask_another_question)
graph.add_edge(START, "ask_question")
graph.add_edge("ask_question", "chatbot")
graph.add_edge("chatbot", "ask_another_question")
graph.add_conditional_edges(source="ask_another_question", path=routing_function)

graph_compiled = graph.compile()
graph_compiled.invoke(MessagesState(messages=[]))

---
## Section 7 — The RemoveMessage Class

### Selective Message Deletion

As conversations grow long, the message list can become very large. This has two problems:
1. **Cost** — longer context means more tokens sent to the LLM API
2. **Context window limits** — LLMs have a maximum number of tokens they can process

**`RemoveMessage`** is a special message type that tells `add_messages` to **delete** a message from the state by its ID, rather than add a new one.

### How It Works
Every message in LangChain has a unique `.id` property (a UUID assigned automatically). To delete a message, you create `RemoveMessage(id=message.id)` and return it from a node. The `add_messages` reducer sees the `RemoveMessage` and removes the matching message from the accumulated list.

This section demonstrates the `RemoveMessage` concept in isolation before embedding it in a graph.

In [ ]:
# Import RemoveMessage — the special message type used for deletions.
from langchain_core.messages import RemoveMessage

In [ ]:
# Build a sample conversation history to demonstrate deletion.
# This simulates a multi-turn conversation after a few exchanges.
my_list = add_messages(
    [
        AIMessage("What is your question?"),
        HumanMessage("Could you tell me a grook by Piet Hein?"),
        AIMessage("Certainly! Here's a well-known grook by Piet Hein..."),
        AIMessage("Would you like to ask one more question?"),
        HumanMessage("yes"),
        AIMessage("What is your question?"),
        HumanMessage("Where was the poet born?"),
        AIMessage("Piet Hein was born in Copenhagen, Denmark, on December 16, 1905."),
        AIMessage("Would you like to ask one more question?"),
    ],
    [HumanMessage("yes")]
)

print(f"Total messages: {len(my_list)}")

In [ ]:
# Show the messages we want to REMOVE (everything except the last 5).
# my_list[:-5] returns all messages from the start up to (but not including) the last 5.
print(f"Messages to remove ({len(my_list[:-5])}):\n")
for msg in my_list[:-5]:
    msg.pretty_print()

In [ ]:
# Create a list of RemoveMessage objects — one per message to delete.
# Each RemoveMessage contains the ID of the message to remove.
# When passed to add_messages(), the reducer handles the deletion.
remove_messages = [RemoveMessage(id=msg.id) for msg in my_list[:-5]]

print(f"Created {len(remove_messages)} RemoveMessage objects.")
print("Sample RemoveMessage:", remove_messages[0])

In [ ]:
# Apply the removals.
# add_messages sees the RemoveMessage objects and deletes the matching messages.
# The result is only the last 5 messages.
trimmed = add_messages(my_list, remove_messages)

print(f"Remaining messages after trimming: {len(trimmed)}\n")
for msg in trimmed:
    msg.pretty_print()

---
## Section 8 — Trimming Messages

### Adding a Trim Node to the Graph

Now we embed the trimming logic into the graph itself. We add a `trim_messages` node that:
1. Looks at the current message list
2. Removes all messages except the most recent 5
3. Returns the `RemoveMessage` objects to trigger the deletions

The routing function is updated to route to `trim_messages` instead of directly back to `ask_question`. After trimming, the graph loops back to `ask_question` to continue the conversation with a lean context window.

### Graph Flow
```
START → ask_question → chatbot → ask_another_question
                                        ↓ (yes)
                               trim_messages
                                        ↓
                               ask_question (loop)
                                        ↓ (no)
                                       END
```

In [ ]:
# Node: trim_messages
# This node doesn't interact with the user or call the LLM.
# Its sole purpose is to prune the message history before the next iteration.
#
# state["messages"][:-5] selects all messages EXCEPT the last 5.
# We create a RemoveMessage for each of those and return them.
# The add_messages reducer will remove those messages from state.
#
# After this node, the state will contain only the 5 most recent messages.
def trim_messages(state: MessagesState) -> MessagesState:
    print("\n-------> ENTERING trim_messages:")
    remove_msgs = [RemoveMessage(id=msg.id) for msg in state["messages"][:-5]]
    return MessagesState(messages=remove_msgs)


# Updated routing function: on 'yes', route to trim_messages first.
def routing_function(state: MessagesState) -> Literal["trim_messages", "__end__"]:
    if state["messages"][-1].content == "yes":
        return "trim_messages"
    else:
        return "__end__"


# Node definitions (same as before)
def ask_question(state: MessagesState) -> MessagesState:
    print("\n-------> ENTERING ask_question:")
    for msg in state["messages"]:
        msg.pretty_print()
    question = "What is your question?"
    print(question)
    return MessagesState(messages=[AIMessage(question), HumanMessage(input())])


def chatbot(state: MessagesState) -> MessagesState:
    print("\n-------> ENTERING chatbot:")
    for msg in state["messages"]:
        msg.pretty_print()
    response = chat.invoke(state["messages"])
    response.pretty_print()
    return MessagesState(messages=[response])


def ask_another_question(state: MessagesState) -> MessagesState:
    print("\n-------> ENTERING ask_another_question:")
    for msg in state["messages"]:
        msg.pretty_print()
    question = "Would you like to ask one more question (yes/no)?"
    print(question)
    return MessagesState(messages=[AIMessage(question), HumanMessage(input())])


# Build the graph with the trim_messages node.
graph = StateGraph(MessagesState)
graph.add_node("ask_question", ask_question)
graph.add_node("chatbot", chatbot)
graph.add_node("ask_another_question", ask_another_question)
graph.add_node("trim_messages", trim_messages)

graph.add_edge(START, "ask_question")
graph.add_edge("ask_question", "chatbot")
graph.add_edge("chatbot", "ask_another_question")
graph.add_conditional_edges(source="ask_another_question", path=routing_function)
# After trimming, loop back to ask for another question.
graph.add_edge("trim_messages", "ask_question")

graph_compiled = graph.compile()
graph_compiled

In [ ]:
# Run the trimming graph.
# Ask a few questions and continue with 'yes' multiple times.
# After each loop, you'll see the message list trimmed to 5 entries.
graph_compiled.invoke(MessagesState(messages=[]))

---
## Section 9 — Summarizing Messages

### Why Summarize Instead of Just Trim?

Trimming discards old messages entirely — the LLM loses context from earlier in the conversation. **Summarizing** offers a better trade-off:

1. The LLM generates a rolling **summary** of the conversation so far
2. All raw messages are deleted to save tokens
3. The summary is injected as a **SystemMessage** on future turns, giving the LLM context without the full message log

### Extended State

We extend `MessagesState` with a `summary: str` field. This is the recommended pattern when you need additional data alongside messages — inherit from `MessagesState` and add your custom fields.

In [ ]:
from langchain_core.messages import SystemMessage

In [ ]:
# Extended State: inherit from MessagesState and add a summary field.
# The summary field stores a rolling text summary of the conversation.
# It is a plain str with no reducer — each update replaces the previous summary.
class State(MessagesState):
    summary: str

In [ ]:
# Test accessing the summary field on an empty state.
# TypedDict does not enforce defaults, so accessing a missing key raises KeyError.
# Use .get() with a default value to safely handle the empty case.
test_state = State()

# This would raise KeyError: test_state["summary"]
# Instead, use .get() with a default:
summary_value = test_state.get("summary", "")
print(f"Summary exists: {bool(summary_value)}")
print(f"Summary value: '{summary_value}'")  # Empty string

In [ ]:
# Node: chatbot (updated for summarization)
# Before calling the LLM, we prepend a SystemMessage containing the conversation summary.
# This gives the LLM context from older messages that were already deleted.
def chatbot(state: State) -> State:
    print("\n-------> ENTERING chatbot:")
    for msg in state["messages"]:
        msg.pretty_print()
    
    # Build a system message from the existing summary (empty string if no summary yet).
    # The f-string template creates a clear context-setting prompt for the LLM.
    system_message = f"""
    Here's a quick summary of what's been discussed so far:
    {state.get("summary", "")}
    
    Keep this in mind as you answer the next question.
    """
    
    # Prepend the system message to the current messages list.
    # SystemMessage sets the behavioral context for the LLM.
    response = chat.invoke([SystemMessage(system_message)] + list(state["messages"]))
    response.pretty_print()
    return State(messages=[response])


# Node: summarize_and_delete_messages
# This node performs two things in one step:
#   1. Generates an updated rolling summary by asking the LLM to extend the previous summary
#   2. Removes ALL current messages from state (they're now captured in the summary)
def summarize_and_delete_messages(state: State) -> State:
    print("\n-------> ENTERING summarize_and_delete_messages:")
    
    # Format the current messages as a readable transcript string.
    new_conversation = ""
    for msg in state["messages"]:
        new_conversation += f"{msg.type}: {msg.content}\n\n"
    
    # Craft a prompt that asks the LLM to update (not repeat) the rolling summary.
    # 'Build upon the previous summary' is key — it prevents redundant repetition.
    summary_instructions = f"""
Update the ongoing summary by incorporating the new lines of conversation below.
Build upon the previous summary rather than repeating it so that the result
reflects the most recent context and developments.

Previous Summary:
{state.get("summary", "")}

New Conversation:
{new_conversation}
"""
    
    print(summary_instructions)
    summary = chat.invoke([HumanMessage(summary_instructions)])
    
    # Delete ALL messages — they are now captured in the summary.
    # state["messages"][:] selects every message in the list.
    remove_messages = [RemoveMessage(id=msg.id) for msg in state["messages"][:]]
    
    # Return both the RemoveMessage deletions and the updated summary.
    return State(messages=remove_messages, summary=summary.content)


# Remaining nodes (unchanged)
def ask_question(state: State) -> State:
    print("\n-------> ENTERING ask_question:")
    question = "What is your question?"
    print(question)
    return State(messages=[AIMessage(question), HumanMessage(input())])


def ask_another_question(state: State) -> State:
    print("\n-------> ENTERING ask_another_question:")
    question = "Would you like to ask one more question (yes/no)?"
    print(question)
    return State(messages=[AIMessage(question), HumanMessage(input())])


def routing_function(state: State) -> Literal["summarize_and_delete_messages", "__end__"]:
    if state["messages"][-1].content == "yes":
        return "summarize_and_delete_messages"
    else:
        return "__end__"


# Build the summarizing graph.
graph = StateGraph(State)
graph.add_node("ask_question", ask_question)
graph.add_node("chatbot", chatbot)
graph.add_node("ask_another_question", ask_another_question)
graph.add_node("summarize_and_delete_messages", summarize_and_delete_messages)

graph.add_edge(START, "ask_question")
graph.add_edge("ask_question", "chatbot")
graph.add_edge("chatbot", "ask_another_question")
graph.add_conditional_edges(source="ask_another_question", path=routing_function)
graph.add_edge("summarize_and_delete_messages", "ask_question")

graph_compiled = graph.compile()
graph_compiled

In [ ]:
# Run the summarizing chatbot.
# Ask a few questions, answer 'yes' to loop.
# After each loop, watch the summary being generated and all messages cleared.
# On subsequent turns, the chatbot uses the summary as context.
graph_compiled.invoke(State(messages=[]))

---
## Section 10 — Short-Term Memory with InMemorySaver

### The Problem: Graph State Resets on Every `.invoke()`

So far, every call to `graph_compiled.invoke()` starts with a fresh empty state. There's no memory between separate invocations.

### The Solution: Checkpointers

A **checkpointer** saves the graph state after every node execution. This enables:
- **Multi-turn conversations** spread across multiple `.invoke()` calls
- **Multiple parallel conversations** (threads) with the same compiled graph
- **State inspection and replay** (covered in Section 11)

### `InMemorySaver`

`InMemorySaver` stores checkpoints in Python dictionaries in RAM. It's ideal for:
- Development and testing
- Short-lived sessions (state is lost when the Python process restarts)

For persistence across restarts, use `SqliteSaver` (covered in Section 12).

### Thread IDs

Each conversation is identified by a `thread_id` in the config dictionary. The same compiled graph can manage multiple independent conversations simultaneously — each thread has its own isolated state.

In [ ]:
# Import InMemorySaver — the in-memory checkpointer for short-term persistence.
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
# Redefine State with summary field.
class State(MessagesState):
    summary: str


# Node: ask_question
def ask_question(state: State) -> State:
    print("\n-------> ENTERING ask_question:")
    question = "What is your question?"
    print(question)
    return State(messages=[AIMessage(question), HumanMessage(input())])


# Node: chatbot (with summary context injection)
def chatbot(state: State) -> State:
    print("\n-------> ENTERING chatbot:")
    system_message = f"""
    Here's a quick summary of what's been discussed so far:
    {state.get("summary", "")}
    
    Keep this in mind as you answer the next question.
    """
    response = chat.invoke([SystemMessage(system_message)] + list(state["messages"]))
    response.pretty_print()
    return State(messages=[response])


# Node: summarize_messages (renamed from previous section for clarity)
def summarize_messages(state: State) -> State:
    print("\n-------> ENTERING summarize_messages:")
    
    new_conversation = ""
    for msg in state["messages"]:
        new_conversation += f"{msg.type}: {msg.content}\n\n"
    
    summary_instructions = f"""
Update the ongoing summary by incorporating the new lines of conversation below.
Build upon the previous summary rather than repeating it,
so that the result reflects the most recent context and developments.
Respond only with the summary.

Previous Summary:
{state.get("summary", "")}

New Conversation:
{new_conversation}
"""
    
    print(summary_instructions)
    summary = chat.invoke([HumanMessage(summary_instructions)])
    remove_messages = [RemoveMessage(id=msg.id) for msg in state["messages"][:]]
    return State(messages=remove_messages, summary=summary.content)


# Build the graph — identical structure to Section 9 but now compiled WITH a checkpointer.
graph = StateGraph(State)
graph.add_node("ask_question", ask_question)
graph.add_node("chatbot", chatbot)
graph.add_node("summarize_messages", summarize_messages)

graph.add_edge(START, "ask_question")
graph.add_edge("ask_question", "chatbot")
graph.add_edge("chatbot", "summarize_messages")
graph.add_edge("summarize_messages", END)

# Create the InMemorySaver checkpointer and pass it to compile().
# From this point on, every node execution is saved as a checkpoint.
checkpointer = InMemorySaver()
graph_compiled = graph.compile(checkpointer)

graph_compiled

In [ ]:
# Define thread configs.
# Each thread_id represents an independent conversation.
# Thread "1" and Thread "2" are completely isolated — they don't share state.
config1 = {"configurable": {"thread_id": "1"}}
config2 = {"configurable": {"thread_id": "2"}}

print("Thread configs defined. Use config1 or config2 when invoking the graph.")

In [ ]:
# Invoke on Thread 2.
# Pass the config as the second argument to .invoke().
# The checkpointer will save the state after each node, keyed to thread_id="2".
#
# State() with no arguments creates an empty state.
# This is equivalent to State(messages=[]) since the checkpointer handles defaults.
graph_compiled.invoke(State(), config2)

In [ ]:
# Invoke AGAIN on Thread 2 — the summary from the previous invocation is loaded.
# The chatbot now has context from the first conversation.
# This demonstrates true multi-turn memory across separate .invoke() calls.
graph_compiled.invoke(State(), config2)

---
## Section 11 — The StateSnapshot Class

### Inspecting Checkpoint History

When a checkpointer is attached, LangGraph saves a **StateSnapshot** after every node execution. These snapshots form a complete audit trail of the graph's execution.

Each `StateSnapshot` contains:
- `values` — the state dictionary at that point in time
- `next` — which node(s) will run next (empty tuple if at END)
- `metadata` — information about the checkpoint (step number, run IDs, etc.)
- `config` — the configuration used for this checkpoint
- `created_at` — timestamp of when the snapshot was created

### Use Cases
- **Debugging** — replay a graph from any historical checkpoint
- **Time travel** — rewind state to a previous point and re-run with different inputs
- **Auditing** — track exactly what happened at each step

In [ ]:
# The graph and checkpointer from Section 10 are still active.
# Run Thread 1 to generate some checkpoint history on a separate thread.
print("Running Thread 1 to generate checkpoint history...")
graph_compiled.invoke(State(), config1)

In [ ]:
# Retrieve the full checkpoint history for Thread 1.
# get_state_history() returns a generator of StateSnapshot objects.
# We convert it to a list so we can index into it.
#
# Note: the history is returned in reverse chronological order
# (most recent checkpoint first). We reverse it for display.
graph_states = list(graph_compiled.get_state_history(config1))
print(f"Total checkpoints saved: {len(graph_states)}")

In [ ]:
# Inspect all snapshots in chronological order (reversed from get_state_history output).
# For each snapshot we print:
#   - Step number (monotonically increasing execution counter)
#   - Messages in state at that point
#   - The summary (if any has been generated)
#   - The 'next' field — which node runs after this checkpoint

for snapshot in graph_states[::-1]:  # Reverse to show chronologically
    print(f"""
--- Step {snapshot.metadata['step']} ---
Messages: {snapshot.values['messages']}
Summary:  {snapshot.values.get('summary', '')}
Next:     {snapshot.next}
""")

---
## Section 12 — Long-Term Memory with SQLite

### Persistence Beyond the Python Session

`InMemorySaver` loses all state when the Python process ends. For true long-term persistence — state that survives kernel restarts, server reboots, or days between conversations — we need a **disk-based checkpointer**.

### `SqliteSaver`

`SqliteSaver` stores checkpoints in an SQLite database file on disk. This means:
- State persists permanently between sessions
- You can restart the notebook, reconnect to the same database, and resume a conversation exactly where it left off
- Multiple threads are stored in the same database, organized by thread ID

### DEPRECATION NOTE
The original course used `from langgraph.checkpoint.sqlite import SqliteSaver`.
This import path is deprecated in newer versions of LangGraph.
The current correct package is `langgraph-checkpoint-sqlite`, installed separately:
```bash
pip install langgraph-checkpoint-sqlite
```
Then import with:
```python
from langgraph.checkpoint.sqlite import SqliteSaver
```
The code below handles both cases gracefully.

In [ ]:
import sqlite3
import os

try:
    # Current recommended import (requires langgraph-checkpoint-sqlite package)
    from langgraph.checkpoint.sqlite import SqliteSaver
    print("SqliteSaver imported successfully.")
except ImportError:
    print("SqliteSaver not available. Install with: pip install langgraph-checkpoint-sqlite")
    print("Falling back to InMemorySaver for this demo.")
    SqliteSaver = None

In [ ]:
# Define State (same as before)
class State(MessagesState):
    summary: str


# Node definitions (same as Section 10)
def ask_question(state: State) -> State:
    print("\n-------> ENTERING ask_question:")
    question = "What is your question?"
    print(question)
    return State(messages=[AIMessage(question), HumanMessage(input())])


def chatbot(state: State) -> State:
    print("\n-------> ENTERING chatbot:")
    system_message = f"""
    Here's a quick summary of what's been discussed so far:
    {state.get("summary", "")}
    
    Keep this in mind as you answer the next question.
    """
    response = chat.invoke([SystemMessage(system_message)] + list(state["messages"]))
    response.pretty_print()
    return State(messages=[response])


def summarize_messages(state: State) -> State:
    print("\n-------> ENTERING summarize_messages:")
    new_conversation = ""
    for msg in state["messages"]:
        new_conversation += f"{msg.type}: {msg.content}\n\n"
    
    summary_instructions = f"""
Update the ongoing summary by incorporating the new lines of conversation below.
Build upon the previous summary rather than repeating it,
so that the result reflects the most recent context and developments.
Respond only with the summary.

Previous Summary:
{state.get("summary", "")}

New Conversation:
{new_conversation}
"""
    print(summary_instructions)
    summary = chat.invoke([HumanMessage(summary_instructions)])
    remove_messages = [RemoveMessage(id=msg.id) for msg in state["messages"][:]]
    return State(messages=remove_messages, summary=summary.content)


# Build the graph
graph = StateGraph(State)
graph.add_node("ask_question", ask_question)
graph.add_node("chatbot", chatbot)
graph.add_node("summarize_messages", summarize_messages)
graph.add_edge(START, "ask_question")
graph.add_edge("ask_question", "chatbot")
graph.add_edge("chatbot", "summarize_messages")
graph.add_edge("summarize_messages", END)

In [ ]:
# Set up the SQLite database.
# IMPORTANT: Update db_path to a location that works for your system.
# On Windows: use raw strings or forward slashes, e.g. r"C:\Users\YourName\langgraph.db"
# On Mac/Linux: e.g. "/home/youruser/langgraph.db" or a relative path like "langgraph.db"
#
# Using a relative path stores the DB in the same directory as this notebook.
db_path = "langgraph.db"

if SqliteSaver is not None:
    # check_same_thread=False is required when sharing the connection across threads
    # (LangGraph may use threading internally for async operations).
    con = sqlite3.connect(database=db_path, check_same_thread=False)
    checkpointer = SqliteSaver(con)
    print(f"SQLite checkpointer connected to: {os.path.abspath(db_path)}")
else:
    # Fallback to InMemorySaver if SqliteSaver is unavailable
    checkpointer = InMemorySaver()
    print("Using InMemorySaver as fallback (no persistence across restarts).")

graph_compiled = graph.compile(checkpointer)
graph_compiled

In [ ]:
# Thread config for the persistent session.
# Use thread_id="1" consistently to resume the same conversation across kernel restarts.
# Change the thread_id to start a fresh conversation.
config1 = {"configurable": {"thread_id": "1"}}

# Run the graph — state will be saved to the SQLite database.
# Restart the kernel, re-run the setup cells, and invoke again with the same config
# to verify that the conversation summary persists.
graph_compiled.invoke(State(), config1)

---
## Summary and Next Steps

Congratulations on completing the LangGraph Learning Path! Here's a recap of what you've covered:

| Concept | Key Takeaway |
|---------|-------------|
| States, Nodes, Edges | The three building blocks of every LangGraph application |
| StateGraph | The builder class — add nodes, add edges, compile |
| Conditional Edges | Routing functions enable branching and loops |
| Reducers (`add_messages`) | Attach merge logic to state fields using `Annotated` |
| `MessagesState` | Built-in state class with `add_messages` pre-configured |
| `RemoveMessage` | Selectively delete messages from accumulated state |
| Trimming | Keep context windows manageable by pruning old messages |
| Summarizing | Preserve context efficiently with rolling LLM-generated summaries |
| `InMemorySaver` | Short-term persistence across multiple `.invoke()` calls |
| `StateSnapshot` | Inspect and replay the full checkpoint history |
| `SqliteSaver` | Long-term persistence that survives kernel/server restarts |

### Suggested Next Steps
- Explore **LangGraph multi-agent systems** (supervisor patterns, handoffs between agents)
- Learn about **human-in-the-loop** workflows using `interrupt_before` and `interrupt_after`
- Study **parallel node execution** using `Send` for fan-out patterns
- Build a complete **RAG + LangGraph** pipeline combining retrieval with stateful conversation